# NB06: 공공시설 (유휴공간 후보) 좌표화 및 통합

**목적**: 드론 스테이션 후보 부지로 활용 가능한 공공시설물을 GeoDataFrame으로 통합

**입력**:
- 주차장 정보 CSV (172개소, 위경도 포함)  
- 주차장 입출차 현황 CSV (유휴율 판별용)  
- 행정복지센터 CSV 3개 (50개소, 주소만 → 지오코딩 필요)

**출력**: `processed/public_facilities.gpkg`

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import requests
import time
from pathlib import Path

BASE = Path(r"C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam")
RAW = BASE / "00_data"
OUT = BASE / "processed"
VWORLD_KEY = "CA689AF7-5C7C-37D4-8973-372AEA113858"

## 1. 주차장 데이터 로드 (위경도 직접 변환)

In [2]:
# 주차장 정보 로드
pk = pd.read_csv(RAW / "경기도_성남시_주차장정보_20251226.csv", encoding="utf-8")
print(f"주차장: {len(pk)}개소")

# GeoDataFrame 변환
gdf_parking = gpd.GeoDataFrame(
    pk,
    geometry=gpd.points_from_xy(pk["경도"], pk["위도"]),
    crs="EPSG:4326",
)

# 핵심 컬럼만 추출
gdf_parking = gdf_parking[[
    "주차장관리번호", "주차장명", "주차장구분", "주차장유형",
    "소재지도로명주소", "주차구획수", "geometry",
]].rename(columns={
    "주차장관리번호": "id", "주차장명": "name", "주차장구분": "type",
    "주차장유형": "subtype", "소재지도로명주소": "address", "주차구획수": "capacity",
})
gdf_parking["facility"] = "주차장"
print(f"유효 좌표: {gdf_parking.geometry.is_valid.sum()}/{len(gdf_parking)}")
gdf_parking.head(3)

주차장: 172개소
유효 좌표: 172/172


,id,name,type,subtype,address,capacity,geometry,facility
0,204-1-000001,수상4,공영,노상,경기도 성남시 수정구 산성대로 415,14,POINT (127.15835 37.44905),주차장
1,204-1-000002,수상5,공영,노상,경기도 성남시 수정구 산성대로 395,17,POINT (127.15764 37.44746),주차장
2,204-1-000003,수상7,공영,노상,경기도 성남시 수정구 산성대로 285,14,POINT (127.14849 37.44166),주차장


## 2. 주차장 유휴율 산출 (입출차 현황 데이터 연동)

In [3]:
# 주차장 입출차 현황 로드
usage = pd.read_csv(
    RAW / "경기도 성남시_공영주차장 월별 입출차 현황_20251210.csv",
    encoding="utf-8",
)
print(f"입출차 현황: {len(usage)} rows")
print(f"컬럼: {usage.columns.tolist()}")

# 연간 평균 입차 대수 계산 (월별 컬럼 합산)
month_cols = [c for c in usage.columns if "월" in c]
usage["annual_avg_entry"] = usage[month_cols].mean(axis=1)

# 주차장명으로 주차장 정보에 조인 → 이용률 계산
usage_summary = (
    usage.groupby("주차장명")
    .agg(avg_monthly_entries=("annual_avg_entry", "mean"))
    .reset_index()
    .rename(columns={"주차장명": "name"})
)

# 주차장 정보에 이용률 병합
gdf_parking = gdf_parking.merge(usage_summary, on="name", how="left")

# 유휴율 = 1 - (이용률/용량) — 이용률 낮을수록 드론 스테이션 부지 후보
gdf_parking["idle_score"] = 1 - (
    gdf_parking["avg_monthly_entries"].fillna(0) 
    / gdf_parking["avg_monthly_entries"].max()
)
print(f"\n유휴율 높은 주차장 (드론 스테이션 후보):")
print(gdf_parking.nlargest(5, "idle_score")[["name", "capacity", "idle_score"]])

입출차 현황: 529 rows
컬럼: ['주차장명', '연도', ' 1월 ', ' 2월 ', ' 3월 ', ' 4월 ', ' 5월 ', ' 6월 ', ' 7월 ', ' 8월 ', ' 9월 ', ' 10월 ', ' 11월 ', ' 12월 ']

유휴율 높은 주차장 (드론 스테이션 후보):
       name  capacity  idle_score
7     수상1-2         7         1.0
9       수상3         7         1.0
16  중상86-87        15         1.0
24  중상22-23        17         1.0
41  중상79상-중       135         1.0


## 3. 행정복지센터 지오코딩 (V-World API)

In [4]:
# 복지센터 3개 구 합치기
wc_files = [
    RAW / "경기도 성남시_분당구홈페이지_행정복지센터 현황_20241024.csv",
    RAW / "경기도 성남시_수정구홈페이지_행정복지센터 현황_20251024.csv",
    RAW / "경기도 성남시_중원구홈페이지_행정복지센터 현황_20251103.csv",
]
dfs = [pd.read_csv(f, encoding="utf-8") for f in wc_files]
df_wc = pd.concat(dfs, ignore_index=True)

# 컬럼명 통일 (데이터기준일 vs 데이터기준일자)
if "데이터기준일자" in df_wc.columns:
    df_wc = df_wc.rename(columns={"데이터기준일자": "데이터기준일"})

print(f"행정복지센터: {len(df_wc)}개소")
print(f"구별: {df_wc['시군구'].value_counts().to_dict()}")

# V-World 지오코딩 함수
def geocode_vworld(address, api_key):
    """V-World 지오코딩 API로 주소 → 위경도 변환"""
    url = "https://api.vworld.kr/req/address"
    params = {
        "service": "address",
        "request": "getcoord",
        "version": "2.0",
        "crs": "epsg:4326",
        "address": address,
        "refine": "true",
        "simple": "false",
        "format": "json",
        "type": "road",
        "key": api_key,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if data["response"]["status"] == "OK":
            point = data["response"]["result"]["point"]
            return float(point["y"]), float(point["x"])  # lat, lon
    except Exception as e:
        print(f"  지오코딩 실패: {address} -> {e}")
    return None, None

# 복지센터 주소 지오코딩
print("\n지오코딩 시작...")
lats, lons = [], []
for _, row in df_wc.iterrows():
    addr = row["주 소"]
    # "경기도" 접두어 없으면 추가
    if not addr.startswith("경기도") and not addr.startswith("성남시"):
        addr = "경기도 " + addr
    if addr.startswith("성남시"):
        addr = "경기도 " + addr
    
    lat, lon = geocode_vworld(addr, VWORLD_KEY)
    lats.append(lat)
    lons.append(lon)
    time.sleep(0.3)  # API 부하 방지

df_wc["lat"] = lats
df_wc["lon"] = lons
success = df_wc["lat"].notna().sum()
print(f"지오코딩 성공: {success}/{len(df_wc)}")

if success < len(df_wc):
    print("\n지오코딩 실패 항목:")
    print(df_wc[df_wc["lat"].isna()][["읍면동", "주 소"]])

행정복지센터: 50개소
구별: {'분당구': 22, '수정구': 17, '중원구': 11}

지오코딩 시작...
  지오코딩 실패: 경기도 성남시 분당구 불정로 312(분당동 42) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EB%B6%84%EB%8B%B9%EA%B5%AC+%EB%B6%88%EC%A0%95%EB%A1%9C+312%28%EB%B6%84%EB%8B%B9%EB%8F%99+42%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 분당구 내정로173번길 21(수내동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EB%B6%84%EB%8B%B9%EA%B5%AC+%EB%82%B4%EC%A0%95%EB%A1%9C173%EB%B2%88%EA%B8%B8+21%28%EC%88%98%EB%82%B4%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 분당구 내정로166번길 17 (수내동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EB%B6%84%EB%8B%B9%EA%B5%AC+%EB%82%B4%EC%A0%95%EB%A1%9C166%EB%B2%88%EA%B8%B8+17+%28%EC%88%98%EB%82%B4%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 분당구 돌마로356번길 43(수내동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EB%B6%84%EB%8B%B9%EA%B5%AC+%EB%8F%8C%EB%A7%88%EB%A1%9C356%EB%B2%88%EA%B8%B8+43%28%EC%88%98%EB%82%B4%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 분당구 황새울로 18번길 14(정자동 147) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EB%B6%84%EB%8B%B9%EA%B5%AC+%ED%99%A9%EC%83%88%EC%9A%B8%EB%A1%9C+18%EB%B2%88%EA%B8%B8+14%28%EC%A0%95%EC%9E%90%EB%8F%99+147%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


...

  지오코딩 실패: 경기도 성남시 중원구 둔촌대로 388, 103호(상대원동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EC%A4%91%EC%9B%90%EA%B5%AC+%EB%91%94%EC%B4%8C%EB%8C%80%EB%A1%9C+388%2C+103%ED%98%B8%28%EC%83%81%EB%8C%80%EC%9B%90%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 중원구 박석로 78(상대원동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EC%A4%91%EC%9B%90%EA%B5%AC+%EB%B0%95%EC%84%9D%EB%A1%9C+78%28%EC%83%81%EB%8C%80%EC%9B%90%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 중원구 둔촌대로 252 (하대원동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EC%A4%91%EC%9B%90%EA%B5%AC+%EB%91%94%EC%B4%8C%EB%8C%80%EB%A1%9C+252+%28%ED%95%98%EB%8C%80%EC%9B%90%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


  지오코딩 실패: 경기도 성남시 중원구 도촌북로 28 (도촌동) -> HTTPSConnectionPool(host='api.vworld.kr', port=443): Max retries exceeded with url: /req/address?service=address&request=getcoord&version=2.0&crs=epsg%3A4326&address=%EA%B2%BD%EA%B8%B0%EB%8F%84+%EC%84%B1%EB%82%A8%EC%8B%9C+%EC%A4%91%EC%9B%90%EA%B5%AC+%EB%8F%84%EC%B4%8C%EB%B6%81%EB%A1%9C+28+%28%EB%8F%84%EC%B4%8C%EB%8F%99%29&refine=true&simple=false&format=json&type=road&key=CA689AF7-5C7C-37D4-8973-372AEA113858 (Caused by SSLError(SSLError(136, '[X509: NO_CERTIFICATE_OR_CRL_FOUND] no certificate or crl found (_ssl.c:4353)')))


지오코딩 성공: 0/50

지오코딩 실패 항목:
            읍면동                                  주 소
0       분당동주민센터              성남시 분당구 불정로 312(분당동 42)
1      수내1동주민센터             성남시 분당구 내정로173번길 21(수내동)
2      수내2동주민센터            성남시 분당구 내정로166번길 17 (수내동)
3      수내3동주민센터             성남시 분당구 돌마로356번길 43(수내동)
4       정자동주민센터        성남시 분당구 황새울로 18번길 14(정자동 147)
5      정자1동주민센터            성남시 분당구 성남대로407번길 12(정자동)
6      정자2동주민센터                  성남시 분당구 내정로 66(정자동)
7      정자3동주민센터                 성남시 분당구 돌마로 242(정자동)
8      서현1동주민센터            성남시 분당구 중앙공원로39번길 35(서현동)
9      서현2동주민센터             성남시 분당구 돌마로476번길 17(서현동)
10     이매1동주민센터           성남시 분당구 성남대로779번길 10 (이매동)
11     이매2동주민센터              성남시 분당구 탄천로79번길 19(이매동)
12     야탑1동주민센터              성남시 분당구 장미로86번길 18(야탑동)
13     야탑2동주민센터               성남시 분당구 야탑로76번길 5(야탑동)
14    야탑3동 주민센터             성남시 분당구 야탑로167번길 12(야탑동)
15      금곡동주민센터           성남시 분당구 미금일로90번길 36-9(구미동)
16      구미동주민센터                 성남시 분당구 구미로 130(구미동)
17     구미1동주민센터    

## 4. 통합 GeoDataFrame 생성 + 저장

In [5]:
# 복지센터 GeoDataFrame 생성 (성공한 것만)
wc_valid = df_wc[df_wc["lat"].notna()].copy()
gdf_welfare = gpd.GeoDataFrame(
    wc_valid,
    geometry=gpd.points_from_xy(wc_valid["lon"], wc_valid["lat"]),
    crs="EPSG:4326",
)
gdf_welfare = gdf_welfare[["읍면동", "주 소", "시군구", "geometry"]].rename(
    columns={"읍면동": "name", "주 소": "address", "시군구": "type"}
)
gdf_welfare["facility"] = "행정복지센터"
gdf_welfare["subtype"] = "행정복지센터"
gdf_welfare["capacity"] = None
gdf_welfare["id"] = [f"WC-{i:03d}" for i in range(len(gdf_welfare))]
gdf_welfare["avg_monthly_entries"] = None
gdf_welfare["idle_score"] = 0.9  # 복지센터 옥상/부지는 기본적으로 유휴 가능

# 통합
gdf_all = pd.concat([
    gdf_parking[["id", "name", "type", "subtype", "address", "capacity", 
                  "facility", "avg_monthly_entries", "idle_score", "geometry"]],
    gdf_welfare[["id", "name", "type", "subtype", "address", "capacity",
                  "facility", "avg_monthly_entries", "idle_score", "geometry"]],
], ignore_index=True)

print(f"통합 공공시설: {len(gdf_all)}개소")
print(f"  주차장: {(gdf_all['facility']=='주차장').sum()}")
print(f"  행정복지센터: {(gdf_all['facility']=='행정복지센터').sum()}")

# 저장
gdf_all = gpd.GeoDataFrame(gdf_all, geometry="geometry", crs="EPSG:4326")
gdf_all.to_file(OUT / "public_facilities.gpkg", driver="GPKG")
print(f"\n저장 완료: {OUT / 'public_facilities.gpkg'}")

통합 공공시설: 172개소
  주차장: 172
  행정복지센터: 0

저장 완료: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\processed\public_facilities.gpkg


C:\Users\jimin\AppData\Local\Temp\ipykernel_53872\2938948740.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gdf_all = pd.concat([


## 5. 시각적 검증 (Folium 지도)

In [6]:
import folium

# 성남시 중심 좌표
center = [37.42, 127.13]
m = folium.Map(location=center, zoom_start=12, tiles="cartodbpositron")

# 성남시 경계 오버레이
boundary = gpd.read_file(OUT / "seongnam_boundary.gpkg", layer="city")
folium.GeoJson(boundary, style_function=lambda x: {
    "fillColor": "none", "color": "black", "weight": 2
}).add_to(m)

# 주차장 (파란 점)
for _, row in gdf_all[gdf_all["facility"] == "주차장"].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3, color="blue", fill=True, fill_opacity=0.6,
        popup=f"{row['name']} (주차: {row['capacity']}면)",
    ).add_to(m)

# 복지센터 (빨간 점)
for _, row in gdf_all[gdf_all["facility"] == "행정복지센터"].iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color="red", fill=True, fill_opacity=0.8,
        popup=f"{row['name']}",
    ).add_to(m)

m